# Universal model evaluation
Load any supported checkpoint through the shared evaluation layer. This notebook is development-only; use `91_final_external_test.ipynb` for the intentional, frozen DDI run.

In [ ]:
from pathlib import Path
import pandas as pd
from src.evaluation.evaluator import (build_evaluation_loader, collect_predictions, evaluate_model, export_predictions, load_checkpoint_bundle, prediction_frame)
from src.evaluation.reporting import compare_experiments, load_experiment_records

STRATEGY = 'efficientnet'
CHECKPOINT = Path('models/efficientnet/replace_me.pt')
CONFIG = Path('configs/efficientnet.yaml')
MANIFEST = Path('data/processed/manifest.csv')
SPLIT = 'validation'
TASK = 'diagnosis_binary'
THRESHOLD = None
CALIBRATION = None
FROZEN_CONFIG = Path('results/final_model/frozen_config.yaml')
BOX_PROVIDER = None  # Required for automatically localized crop strategies.

In [ ]:
records = load_experiment_records()
display(compare_experiments(records, task=TASK))
bundle = load_checkpoint_bundle(CHECKPOINT, strategy=STRATEGY, config_path=CONFIG)
assert bundle.task == TASK, (bundle.task, TASK)
manifest = pd.read_csv(MANIFEST)
assert not manifest.dataset.fillna('').str.upper().eq('DDI').any(), 'Use 91_final_external_test.ipynb for DDI.'
loader = build_evaluation_loader(manifest, bundle, split=SPLIT, box_provider=BOX_PROVIDER)
metrics = evaluate_model(bundle.model, loader, task=TASK, class_order=bundle.class_order, threshold=THRESHOLD, calibration=CALIBRATION, evaluation_split=SPLIT)
metrics

In [ ]:
collected = collect_predictions(bundle.model, loader, class_order=bundle.class_order)
probabilities = collected['probabilities'] if CALIBRATION is None else CALIBRATION.apply(collected['logits'])
predictions = prediction_frame(collected['targets'], probabilities, collected['metadata'], class_order=bundle.class_order, task=TASK, strategy=bundle.strategy, checkpoint_id=CHECKPOINT.name, threshold=THRESHOLD, calibrated=CALIBRATION is not None)
output_dir = Path('results/evaluations')
export_predictions(predictions, output_dir / f'{CHECKPOINT.stem}_{SPLIT}_predictions.csv')
predictions.head()